# Demo E: Vienna MRI with Polaris respiratory tracking

Reconstruct zero-based slice 15 using the largest-amplitude Polaris tool axis, filtered
with a 1.0 Hz low-pass. Tracking and MRI are aligned at the full sequence end;
tracking coordinates are physiological surrogates in export units, not MRI-axis vectors.
Set the local paths below. Outputs include raw/filtered XYZ overlays and the same
input and reconstruction panels as Demo B.


In [ ]:
import time
from pathlib import Path
import matplotlib.pyplot as plt

from src.runtime.runtime_config import load_config
from src.runtime.runtime_setup import initialize_runtime
from src.preprocessing.DataLoader import DataLoader
from src.preprocessing.physiological_data.PolarisInfraredTrackerReader import PolarisInfraredTrackerReader
from src.reconstruction.JointReconstructor import JointReconstructor
from src.utils.notebook_display import display_input_sampling_motion_panels, display_run_panels

jupyter_notebook_flag = True
DATA_ROOT = Path.cwd().parent / "data" / "GRICS-torch" / "Vienna"
MOTION_FILE = DATA_ROOT / "motion/R1.tsv"
RAW_FILE = DATA_ROOT / "raw_data/meas_MID01133_FID01857_AX_T2_TSE_HR_2NEX_SAT_MOTION.dat"
OUTPUT_ROOT = Path("results/demo_e_vienna_slice15_existing_reader")
SLICE_IDX = 15  # zero-based source slice
POLARIS_CHANNEL_MODE = "largest-amplitude"  # "all" uses XYZ


def main():
    params = load_config(
        data_type="siemens-polaris",
        reconstruction_config="config/reconstruction/nonrigid_2d.toml",
        simulated_motion_type="as-it-is", data_dimension="2D",
        overrides={
            "jupyter_notebook_flag": jupyter_notebook_flag,
            "clean_output_folders_before_run": False,
            "debug_folder": str(OUTPUT_ROOT / "debug"),
            "logs_folder": str(OUTPUT_ROOT / "logs"),
            "results_folder": str(OUTPUT_ROOT / "reconstruction"),
            "initial_data_folder": str(OUTPUT_ROOT / "input"),
        },
    )
    sp_device, t_device = initialize_runtime(params)

    print("[Demo E] Preparing MRI and Polaris data...")
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    data = DataLoader(
        params=params, t_device=t_device, sp_device=sp_device,
        filename=(str(RAW_FILE), str(MOTION_FILE)), slice_idx=SLICE_IDX,
        polaris_channel_mode=POLARIS_CHANNEL_MODE, run_pipeline=False,
    )
    data.load_data()
    preparer = data.raw_data_preparer
    print("Polaris motion channels:", preparer.selected_polaris_channels)

    # Compare raw and filtered positions on the full-sequence clock.
    tracking = PolarisInfraredTrackerReader.read(MOTION_FILE)
    sync = preparer.synchronization
    selected = sync["slice_indices"] == SLICE_IDX
    fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True, constrained_layout=True)
    for axis, label in enumerate("XYZ"):
        axes[axis].plot(tracking.time_seconds - tracking.time_seconds[-1],
                        tracking.tool_positions[:, axis], label="Raw R1", color="gray", alpha=0.6)
        axes[axis].plot(sync["physiological_time_seconds"][axis], sync["physiological_values"][axis],
                        label="Filtered R1 (low-pass 1.0 Hz)", color="tab:blue")
        axes[axis].scatter(sync["acquisition_time_seconds"][selected],
                           sync["acquisition_values"][selected, axis], s=5, label="Slice readouts")
        axes[axis].axvline(0, color="black", linestyle="--", label="Sequence end")
        axes[axis].set_ylabel(f"{label} [export units]")
        axes[axis].grid(alpha=0.3)
    axes[0].legend()
    axes[-1].set_xlabel("Time relative to full sequence end [s]")
    fig.savefig(OUTPUT_ROOT / "synchronization.png", dpi=150)
    plt.show()

    print("[Demo E] Loading data and building operators...")
    data.run_slice_pipeline()
    display_input_sampling_motion_panels(params)
    recon = JointReconstructor(
        data.kspace, data.smaps, data.sampling_idx, motion_signal=data.motion_signal,
        params=params, motion_plot_context=data.motion_plot_context,
    )
    print("[Demo E] Starting reconstruction...")
    t0 = time.time()
    recon.run()
    print(f"Elapsed time: {time.time() - t0:.2f} s")
    display_run_panels(params, motion_type=params.reconstruction_motion_type)


if __name__ == "__main__":
    main()
